# Riot API Data Collector

In [1]:
import requests, time, json, datetime
import pandas as pd
from tqdm.notebook import tqdm
from config import *
import random

In [2]:
# ── API 工具函数 ───────────────────────────────────
def api_get(url, params=None, retries=6):
    for attempt in range(retries):
        res = requests.get(url, headers=HEADERS, params=params or {})
        if res.status_code == 200:
            return res.json()
        elif res.status_code == 429:
            wait = int(res.headers.get("Retry-After", 2 ** (attempt + 1)))
            print(f"  [429] waiting {wait}s …")
            time.sleep(wait)
        elif res.status_code == 404:
            return None
        else:
            time.sleep(1.5)
    return None

def fetch_puuids_for_tier(tier: str, n_per_division: int) -> list:
    print(f"  Fetching {tier} …")
    puuids, seen = [], set()

    for div in ["I", "II", "III", "IV"]:
        collected = 0
        page = 1

        while collected < n_per_division:
            url = f"https://{REGION}.api.riotgames.com/lol/league/v4/entries/RANKED_SOLO_5x5/{tier}/{div}"
            entries = api_get(url, params={"page": page})
            time.sleep(0.6)

            if not entries:  # 没数据了就停
                break

            random.shuffle(entries)

            for entry in entries:
                if collected >= n_per_division:
                    break
                puuid = entry.get("puuid")
                if puuid and puuid not in seen:
                    puuids.append(puuid)
                    seen.add(puuid)
                    collected += 1

            page += 1

        print(f"    {tier} {div}: {collected} players")

    print(f"  ✓ {len(puuids)} PUUIDs collected for {tier}")
    return puuids

def get_match_ids_pre(puuid, count=20):
    url = f"https://{ROUTE}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
    data = api_get(url, params={"start": 0, "startTime": 1758513600, "endTime": 1761105600,"count": count})
    time.sleep(0.6)
    return data if isinstance(data, list) else []

def get_match_ids_post(puuid, count=20):
    url = f"https://{ROUTE}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
    data = api_get(url, params={"start": 0, "startTime": 1761105600, "endTime": 1763787600,"count": count})
    time.sleep(0.6)
    return data if isinstance(data, list) else []

def get_match_data(match_id):
    url = f"https://{ROUTE}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    data = api_get(url)
    time.sleep(0.6)
    if data is None:
        return None
    return data


  TIER: IRON
  Fetching IRON …
    IRON I: 0 players


In [4]:
import datetime

def to_timestamp(date_str):
    dt = datetime.datetime.strptime(date_str, "%Y-%m-%d")
    return int(dt.timestamp())

print(to_timestamp("2025-10-22"))
print(to_timestamp("2025-09-22"))
print(to_timestamp("2025-11-22"))

1761105600
1758513600
1763787600


In [ ]:
# ── 主循环 ─────────────────────────────────────────
match_pre  = []
match_post  = []  

for tier in TIERS:
    print(f"\n{'='*50}\n  TIER: {tier}\n{'='*50}")
    puuids = fetch_puuids_for_tier(tier, n_per_division=SAMPLES_PER_DIVISION)

    for puuid in tqdm(puuids, desc=f"{tier} players"):
        
        match_ids_pre = get_match_ids_pre(puuid, MATCHES_PER_PLAYER)
        match_ids_post = get_match_ids_post(puuid, MATCHES_PER_PLAYER)
        if not match_ids_pre:
            continue
        if not match_ids_post:
            continue


        for mid in match_ids_pre:
            data = get_match_data(mid)
            if data is None:
                continue

            player_total += 1
            match_pre.append(data)

        for mid in match_ids_post:
            data = get_match_data(mid)
            if data is None:
                continue

            player_total += 1
            match_post.append(data)


    # ✅ 每个 tier 跑完立刻存一次，防止崩了全没
    with open(f"raw_valid_{tier}.json", "w") as f:
        json.dump(raw_valid, f)
    print(f"  ✅ {tier} saved → raw_valid_{tier}.json")

    with open(f"raw_total_{tier}.json", "w") as f:
        json.dump(raw_total, f)
    print(f"  ✅ {tier} saved → raw_total_{tier}.json")

# ── 汇总 ───────────────────────────────────────────
total_all = sum(r["total"] for r in all_stats)
valid_all = sum(r["valid"] for r in all_stats)

print(f"\n====== FINAL RESULT ======")
print(f"Total matches:  {total_all}")
print(f"Before Oct 22:  {valid_all}")
print(f"Keep ratio:     {round(valid_all/total_all, 3) if total_all else 0}")

# ── 存全量 JSON ────────────────────────────────────
with open("raw_valid_all.json", "w") as f:
    json.dump(raw_valid, f)
with open("all_stats.json", "w") as f:
    json.dump(all_stats, f)

print("✅ Done. raw_valid_all.json / all_stats.json saved.")